In [ ]:
# Pin here for fix of CPU offloading bug; implemented in docker /uv
#%pip install "transformers==4.57.3" "accelerate==1.12.0" "bitsandbytes==0.49.1" "vllm<0.22.0"

In [ ]:
from experiments_vllm import *
from data import *
from config_eplb import *

In [ ]:
# Environment fixes
import os
os.environ["FLASHINFER_DISABLE_VERSION_CHECK"] = "1"

import sys, os
venv_bin = os.path.dirname(sys.executable)
if venv_bin not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = venv_bin + os.pathsep + os.environ["PATH"]

In [ ]:
# Set seeds
seed = 43
torch.manual_seed(seed);

In [ ]:
# Use MMLU questions
dataset = get_data_mmlu(n_samples=n_samples, shuffle_seed=seed)
mmlu_prompts, subjects, questions = format_prompts_mmlu(dataset)

In [ ]:
# Make trace directories
if not os.path.isdir(trace_path_eplb):
    os.mkdir(trace_path_eplb)
if not os.path.isdir(trace_path_noeplb):
    os.mkdir(trace_path_noeplb)

In [ ]:
# Measure throughput and record traces with eplb disabled
results_noeplb = await measure_vllm_throughput(model_id,
                                             mmlu_prompts,
                                             seed=seed,
                                             max_new_tokens=max_new_tokens,
                                             max_model_len=max_model_len,
                                             batch_size=batch_size,
                                             concurrency_limit=batch_size*4,
                                             gpu_memory_utilization=gpu_memory_utilization,
                                             n_gpus=n_gpus,
                                             n_warmup_samples=n_warmup_samples,
                                             print_output=False,
                                             enable_expert_parallel=enable_expert_parallel,
                                             enable_prefix_caching=enable_prefix_caching,
                                             enable_eplb=False,
                                             trace_dir=trace_path_noeplb)

In [ ]:
# Measure throughput and record traces with eplb enabled
results_eplb = await measure_vllm_throughput(model_id,
                                             mmlu_prompts,
                                             seed=seed,
                                             max_new_tokens=max_new_tokens,
                                             max_model_len=max_model_len,
                                             batch_size=batch_size,
                                             concurrency_limit=batch_size*4,
                                             gpu_memory_utilization=gpu_memory_utilization,
                                             n_gpus=n_gpus,
                                             n_warmup_samples=n_warmup_samples,
                                             print_output=False,
                                             enable_expert_parallel=enable_expert_parallel,
                                             enable_prefix_caching=enable_prefix_caching,
                                             enable_eplb=True,
                                             trace_dir=trace_path_eplb)

In [ ]:
# Save results
"""
import pickle
with open(results_file_balanced, 'wb') as file:
    pickle.dump(balanced_results, file)
with open(results_file_imbalanced, 'wb') as file:
    pickle.dump(imbalanced_results, file)
"""